Evo finalnog prijedloga — jednostavno i jasno.

Koristi sequence-aware XGBoost (poslednji kod koji sam ti dao).
Razlog je prost — imaš sve što taj model treba:

✅ Datum kupovine svake polise
✅ Kvalitetne demografske podatke
✅ Sumu osiguranja i premiju po polisi
✅ 53k klijenata — dovoljno za treniranje


Zaboravi na Collaborative Filtering iz prethodnih verzija. CF bi bio bolji jedino da nemaš dobre demografske podatke ni datume. Ti imaš oboje, pa nema smisla koristiti slabiji pristup.

Jedina stvar koju dodaj uz XGBoost je jednostavno pravilo za klijente koji imaju samo 1 polisu (za njih ne možeš graditi historiju):
python# Za klijente s 1 polisom — preporuči najpopularnije 
# vrste polisa kod klijenata sličnog profila (starost, pol, djelatnost)
To pokrivaš sa 10-ak linija koda kao fallback, nije poseban model.

Redoslijed koraka:

Izvuci čiste podatke sa datumom, sumom, premijom
Pokreni run_pipeline() iz poslednjeg koda
Pogledaj Hit Rate i Mean AUC iz evaluacije
Ako je Hit Rate ispod 20% — javi mi pa podešavamo feature-e
Ako je iznad 20% — model je spreman za produkciju

In [ ]:
"""
XGBoost model za preporuku osiguranja - sequence-aware pristup
==============================================================
Za svakog klijenta:
  - Sortiramo polise po datumu kupovine
  - Zadnja polisa = TARGET (šta predviđamo)
  - Sve prethodne polise = FEATURE-I (historija)
  
Feature-i po klijentu:
  - Za svaku vrstu polise: ima/nema + suma_osiguranja + premija (iz historije)
  - Demografija: starost, pol, djelatnost, grad
  - Agregirani: ukupna_premija, ukupna_suma, broj_polisa, ima_stetu
"""

import pandas as pd
import numpy as np
import hashlib
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score


# ===========================================================================
# KORAK 1: Učitavanje i čišćenje
# ===========================================================================

def load_and_clean(path_polise: str, path_klijenti: str,
                   datum_col: str = 'datum_pocetka') -> pd.DataFrame:
    """
    Učitaj i spoji podatke. Vrati jedan red po (klijent, polisa).
    
    datum_col: naziv kolone s datumom kupovine polise u df_polise
    """
    df_polise   = pd.read_csv(path_polise,   parse_dates=[datum_col])
    df_klijenti = pd.read_csv(path_klijenti)

    # Anonimizacija JMBG
    df_klijenti['klijent_id'] = df_klijenti['osig_jmbg'].astype(str).apply(
        lambda x: hashlib.sha256(x.encode()).hexdigest()[:12]
    )

    # Starost iz JMBG
    def jmbg_to_age(jmbg):
        try:
            jmbg = str(jmbg).zfill(13)
            god  = int(jmbg[4:7])
            god  = (2000 + god) if god < 900 else (1000 + god)
            from datetime import date
            age  = date.today().year - god
            return age if 16 <= age <= 100 else np.nan
        except:
            return np.nan

    df_klijenti['starost'] = df_klijenti['osig_jmbg'].apply(jmbg_to_age)

    # Uzmi najnoviji demografski zapis po klijentu
    demo = (df_klijenti
            .sort_values(datum_col, ascending=False)
            .drop_duplicates('klijent_id')[
                ['klijent_id', 'starost', 'pol_id', 'sif_delatnost', 'osig_mesto']
            ])

    # Spoji polise sa klijent_id
    df = df_polise.merge(
        df_klijenti[['ponuda_id', 'klijent_id']].drop_duplicates(),
        on='ponuda_id', how='left'
    )

    # Ukloni nevažeće
    df = df[
        df['klijent_id'].notna() &
        df['sif_vrsta'].notna() &
        (df['sif_vrsta'] != 0) &
        (df['konacna_premija'] > 0)
    ].copy()

    # Sortiraj po klijentu i datumu
    df = df.sort_values(['klijent_id', datum_col])

    # Spoji demografiju
    df = df.merge(demo, on='klijent_id', how='left')

    print(f"✓ Učitano: {len(df)} polisa, {df['klijent_id'].nunique()} klijenata")
    print(f"  Vrste polisa: {sorted(df['sif_vrsta'].unique().tolist())}")

    return df


# ===========================================================================
# KORAK 2: Izgradnja sequence-aware dataset-a
# ===========================================================================

def build_sequence_dataset(df: pd.DataFrame,
                            suma_col: str = 'suma_osiguranja') -> pd.DataFrame:
    """
    Za svakog klijenta s 2+ polise:
      - Zadnja polisa (po datumu) = target
      - Sve prethodne = feature-i u wide formatu
    
    Vraća jedan red po klijentu.
    
    Kolone rezultata:
      vrsta_{X}_ima, vrsta_{X}_suma, vrsta_{X}_premija  (za svaku vrstu)
      + demografija + agregirani feature-i
      + target_vrsta (ono što predviđamo)
    """

    # Provjeri postoji li kolona za sumu
    has_suma = suma_col in df.columns
    if not has_suma:
        print(f"⚠ Kolona '{suma_col}' nije pronađena — koristim premiju×10 kao proxy")
        df = df.copy()
        df[suma_col] = df['konacna_premija'] * 10

    sve_vrste = sorted(df['sif_vrsta'].unique().tolist())
    rows = []

    for klijent_id, grupa in df.groupby('klijent_id'):
        grupa = grupa.sort_values('datum_pocetka')

        # Treba min 2 polise
        if len(grupa) < 2:
            continue

        # Zadnja = target
        target_vrsta = grupa.iloc[-1]['sif_vrsta']
        historija    = grupa.iloc[:-1]  # sve osim zadnje

        # --- Feature-i iz historije (wide format) ---
        row = {'klijent_id': klijent_id}

        for vrsta in sve_vrste:
            polise_vrste = historija[historija['sif_vrsta'] == vrsta]
            if len(polise_vrste) > 0:
                row[f'vrsta_{vrsta}_ima']     = 1
                row[f'vrsta_{vrsta}_suma']    = polise_vrste[suma_col].sum()
                row[f'vrsta_{vrsta}_premija'] = polise_vrste['konacna_premija'].sum()
            else:
                row[f'vrsta_{vrsta}_ima']     = 0
                row[f'vrsta_{vrsta}_suma']    = 0
                row[f'vrsta_{vrsta}_premija'] = 0

        # --- Agregirani feature-i iz historije ---
        row['ukupna_premija']         = historija['konacna_premija'].sum()
        row['prosjecna_premija']      = historija['konacna_premija'].mean()
        row['ukupna_suma']            = historija[suma_col].sum()
        row['broj_polisa_historija']  = len(historija)
        row['broj_razlicitih_vrsta']  = historija['sif_vrsta'].nunique()
        row['ima_stetu']              = int((historija.get('ind_steta', pd.Series()) == 'D').any())

        # --- Demografija ---
        row['starost']       = grupa.iloc[0]['starost']
        row['pol_id']        = grupa.iloc[0]['pol_id']
        row['sif_delatnost'] = grupa.iloc[0]['sif_delatnost']
        row['osig_mesto']    = grupa.iloc[0]['osig_mesto']

        # --- Target ---
        row['target_vrsta'] = target_vrsta

        rows.append(row)

    dataset = pd.DataFrame(rows)
    print(f"\n✓ Sequence dataset: {len(dataset)} klijenata (sa 2+ polise)")
    print(f"  Feature kolone: {len([c for c in dataset.columns if c.startswith('vrsta_')])}"
          f" (wide) + {len([c for c in dataset.columns if not c.startswith('vrsta_') and c not in ['klijent_id','target_vrsta']])} (ostali)")

    return dataset, sve_vrste


# ===========================================================================
# KORAK 3: Enkodiranje feature-a
# ===========================================================================

def encode_features(dataset: pd.DataFrame,
                    sve_vrste: list,
                    fit: bool = True,
                    encoders: dict = None) -> tuple:
    """
    Pripremi X matricu za XGBoost.
    Vraća (X, feature_names, encoders).
    """
    data = dataset.copy()

    if encoders is None:
        encoders = {'label': {}, 'scaler': StandardScaler()}

    # Kategoričke kolone
    cat_cols = ['pol_id', 'sif_delatnost', 'osig_mesto']
    for col in cat_cols:
        if col not in data.columns:
            data[col] = 0
            continue
        data[col] = data[col].fillna('NEPOZNATO').astype(str)
        if fit:
            le = LabelEncoder()
            data[col] = le.fit_transform(data[col])
            encoders['label'][col] = le
        else:
            le = encoders['label'].get(col)
            if le:
                known = set(le.classes_)
                data[col] = data[col].apply(lambda x: x if x in known else 'NEPOZNATO')
                data[col] = le.transform(data[col])
            else:
                data[col] = 0

    # Sve numeričke kolone osim ID-a i targeta
    exclude = {'klijent_id', 'target_vrsta'}
    feature_cols = [c for c in data.columns if c not in exclude]

    for col in feature_cols:
        data[col] = pd.to_numeric(data[col], errors='coerce').fillna(0)

    X = data[feature_cols].values

    if fit:
        X = encoders['scaler'].fit_transform(X)
    else:
        X = encoders['scaler'].transform(X)

    return X, feature_cols, encoders


# ===========================================================================
# KORAK 4: XGBoost model (jedan po vrsti polise)
# ===========================================================================

class SequenceXGBoostRecommender:
    """
    Za svaku vrstu polise trenira XGBoost koji odgovara:
    'Da li će ovaj klijent, na osnovu dosadašnje historije, 
     sljedeće kupiti ovu vrstu polise?'
    """

    def __init__(self):
        self.models       = {}
        self.encoders     = None
        self.feature_cols = None
        self.sve_vrste    = None

    def fit(self, dataset: pd.DataFrame, sve_vrste: list):
        self.sve_vrste = sve_vrste

        X, self.feature_cols, self.encoders = encode_features(
            dataset, sve_vrste, fit=True
        )

        print(f"\n⏳ Treniram XGBoost za {len(sve_vrste)} vrsta polisa...")

        for vrsta in sve_vrste:
            # Binary target: je li zadnja kupljena polisa ove vrste?
            y      = (dataset['target_vrsta'] == vrsta).astype(int).values
            n_pos  = y.sum()

            if n_pos < 15:
                print(f"   ⚠ Preskačem vrstu {vrsta} — samo {n_pos} pozitivnih primjera")
                continue

            n_neg = len(y) - n_pos
            scale = round(n_neg / n_pos, 2)

            clf = XGBClassifier(
                n_estimators      = 300,
                max_depth         = 4,
                learning_rate     = 0.05,
                subsample         = 0.8,
                colsample_bytree  = 0.8,
                scale_pos_weight  = scale,
                use_label_encoder = False,
                eval_metric       = 'auc',
                random_state      = 42,
                verbosity         = 0
            )
            clf.fit(X, y)
            self.models[vrsta] = clf
            print(f"   ✓ Vrsta {vrsta}: {n_pos} pos / {n_neg} neg (scale={scale})")

        print(f"\n✓ Trenirano {len(self.models)} modela.")
        return self

    def predict_scores(self, dataset_row: pd.DataFrame) -> pd.Series:
        """Vrati vjerovatnoću za svaku vrstu polise."""
        X, _, _ = encode_features(
            dataset_row, self.sve_vrste, fit=False, encoders=self.encoders
        )
        scores = {}
        for vrsta, model in self.models.items():
            scores[vrsta] = round(model.predict_proba(X)[0][1], 4)
        return pd.Series(scores).sort_values(ascending=False)

    def recommend(self, dataset_row: pd.DataFrame,
                  already_has: list = None, top_n: int = 5) -> pd.DataFrame:
        """
        Preporuka za jednog klijenta.
        already_has: lista vrsta koje klijent već ima (izbacujemo ih)
        """
        scores = self.predict_scores(dataset_row)

        if already_has:
            scores = scores[~scores.index.isin(already_has)]

        top = scores.nlargest(top_n).reset_index()
        top.columns = ['vrsta_polise', 'score']
        top['rank'] = range(1, len(top) + 1)
        return top[['rank', 'vrsta_polise', 'score']]

    def feature_importance(self, vrsta) -> pd.DataFrame:
        """Koje feature-i su najvažniji za datu vrstu polise."""
        if vrsta not in self.models:
            return pd.DataFrame()
        imp = pd.DataFrame({
            'feature'   : self.feature_cols,
            'importance': self.models[vrsta].feature_importances_
        }).sort_values('importance', ascending=False)
        return imp.head(15)


# ===========================================================================
# KORAK 5: Evaluacija
# ===========================================================================

def evaluate(model: SequenceXGBoostRecommender,
             test_dataset: pd.DataFrame,
             test_sve_vrste: list,
             top_n: int = 5) -> dict:
    """
    Evaluacija: da li je model pogodio stvarnu zadnju kupljenu polisu
    u top_n preporukama?
    """
    hits  = 0
    total = 0
    auc_scores = []

    X_test, _, _ = encode_features(
        test_dataset, test_sve_vrste, fit=False, encoders=model.encoders
    )

    for vrsta, clf in model.models.items():
        y_true = (test_dataset['target_vrsta'] == vrsta).astype(int).values
        if y_true.sum() == 0:
            continue
        y_prob = clf.predict_proba(X_test)[:, 1]
        try:
            auc = roc_auc_score(y_true, y_prob)
            auc_scores.append(auc)
        except:
            pass

    # Hit Rate
    for _, row in test_dataset.iterrows():
        stvarna_vrsta = row['target_vrsta']
        row_df = pd.DataFrame([row])
        
        # Polise koje ima u historiji
        already_has = [v for v in test_sve_vrste
                       if row.get(f'vrsta_{v}_ima', 0) == 1]

        recs = model.recommend(row_df, already_has=already_has, top_n=top_n)
        if stvarna_vrsta in recs['vrsta_polise'].values:
            hits += 1
        total += 1

    hit_rate = hits / total if total > 0 else 0
    mean_auc = np.mean(auc_scores) if auc_scores else 0

    print(f"\n📊 Evaluacija (top {top_n}):")
    print(f"   Hit Rate:   {hit_rate:.2%}  ({hits}/{total} klijenata)")
    print(f"   Mean AUC:   {mean_auc:.4f}  (po modelu, prosjek)")
    print(f"   Slučajno:   ~{top_n / len(test_sve_vrste):.2%}")

    return {'hit_rate': hit_rate, 'mean_auc': mean_auc, 'hits': hits, 'total': total}


# ===========================================================================
# KORAK 6: Glavni pipeline
# ===========================================================================

def run_pipeline(path_polise: str, path_klijenti: str,
                 datum_col: str = 'datum_pocetka', top_n: int = 5):

    print("=" * 60)
    print("SEQUENCE-AWARE XGBOOST PREPORUKA OSIGURANJA")
    print("=" * 60)

    # Učitaj i očisti
    df = load_and_clean(path_polise, path_klijenti, datum_col=datum_col)

    # Izgradi sequence dataset
    dataset, sve_vrste = build_sequence_dataset(df)

    # Train/test split
    train_ds, test_ds = train_test_split(dataset, test_size=0.2, random_state=42)
    print(f"\n  Train: {len(train_ds)} | Test: {len(test_ds)}")

    # Treniraj model
    model = SequenceXGBoostRecommender()
    model.fit(train_ds, sve_vrste)

    # Evaluacija
    evaluate(model, test_ds, sve_vrste, top_n=top_n)

    # Primjer preporuke
    primjer = train_ds.iloc[[0]]
    klijent_id = primjer['klijent_id'].values[0]
    already_has = [v for v in sve_vrste if primjer[f'vrsta_{v}_ima'].values[0] == 1]
    
    print(f"\n🎯 Primjer — klijent {klijent_id}:")
    print(f"   Historija vrsta: {already_has}")
    print(f"   Stvarna zadnja:  {primjer['target_vrsta'].values[0]}")
    print(f"   Preporuke:")
    print(model.recommend(primjer, already_has=already_has, top_n=top_n).to_string(index=False))

    # Feature importance za jednu vrstu
    prva_vrsta = list(model.models.keys())[0]
    print(f"\n📈 Top feature-i za vrstu {prva_vrsta}:")
    print(model.feature_importance(prva_vrsta).to_string(index=False))

    # Export svih preporuka
    print("\n⏳ Generišem preporuke za sve klijente...")
    sve_preporuke = []
    for _, row in dataset.iterrows():
        row_df      = pd.DataFrame([row])
        already_has = [v for v in sve_vrste if row.get(f'vrsta_{v}_ima', 0) == 1]
        recs        = model.recommend(row_df, already_has=already_has, top_n=top_n)
        recs['klijent_id'] = row['klijent_id']
        sve_preporuke.append(recs)

    output = pd.concat(sve_preporuke, ignore_index=True)
    output.to_csv('preporuke_sequence_output.csv', index=False)
    print(f"✓ Sačuvano u 'preporuke_sequence_output.csv' — {len(output)} redova")

    return model, dataset


# ===========================================================================
# Pokretanje
# ===========================================================================

if __name__ == "__main__":
    model, dataset = run_pipeline(
        path_polise   = "polise.csv",
        path_klijenti = "klijenti.csv",
        datum_col     = "datum_pocetka",  # ← promijeni u naziv tvoje datum kolone
        top_n         = 5
    )